In [1]:
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

/home/alejo/proyectos/Agente-Consulta-Paginas-Web/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# URLs de las páginas web a consultar
urls = [
    "https://seaborn.pydata.org/",
    "https://matplotlib.org/",
    "https://plotly.com/python/"
]

# Función para obtener contenido
def get_page_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    texts = soup.stripped_strings
    return ' '.join(texts)

# Obtener textos
texts = [get_page_text(url) for url in urls]

# Cargar modelo y tokenizer
model_name = "google/flan-t5-base" #"google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Función para resumir
def summarize(text, max_length=150):
    inputs = tokenizer.encode("Summary: " + text, return_tensors="pt", max_length=10000, truncation=True)
    summary_ids = model.generate(inputs, max_length=max_length, num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Resumir páginas
summaries = [summarize(text) for text in texts]


C:\Users\46062352\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\46062352\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
summaries[1]

'Summary: Matplotlib: Visualization with Python Skip to main content Ctrl + K Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Matplotlib 3.9.0 Released We thank the 175 authors for the 450 pull requests that comprise the 3.9.0 release. Older Announcements Resources # Be sure to check the Users guide and the API docs . The full text search is a good way to discover the docs including the many examples. Join our community at discourse.matplotlib.'

In [5]:
# Crear el prompt para comparación
prompt = (
"Below are summaries of three visualization libraries:\n\n"
f"Summary 1: {texts[0], summaries[0]}\n"
f"Summary 2: {texts[1], summaries[1]}\n"
f"Summary 3: {texts[2], summaries[2]}\n\n"
"Based on these summaries, provide an objective comparison and recommend which of these libraries is best for general-purpose use. Create a comparison table considering the following criteria: Ease of use / learning curve, Default visual aesthetics, Customization capabilities, Level of interactivity, Support for statistical analysis, and Integration with other tools (such as Jupyter, Dash, etc.). Additionally, give a brief recommendation on when to use each one, considering ease of use, size, and applications. Explain your choice.\n\nAnswer:"
)

prompt = f"""
Compare objectively the following three data visualization libraries based on their features.

Summary 1 (Seaborn): {texts[0]}
Summary 2 (Matplotlib): {texts[1]}
Summary 3 (Plotly): {texts[2]}

Create a markdown table with the following criteria:
- Ease of use / learning curve
- Default visual aesthetics
- Customization capabilities
- Level of interactivity
- Support for statistical analysis
- Integration with other tools (e.g., Jupyter, Dash)

Then provide:
1. A recommendation of when to use each library.
2. A final recommendation on which is best for general-purpose use.

Answer in a structured and clear format.
"""


In [10]:
# Generar la comparación
inputs = tokenizer.encode(prompt, return_tensors="pt", max_length=1024, truncation=True)
response_ids = model.generate(inputs, max_length=300, num_beams=4, early_stopping=True)
comparison = tokenizer.decode(response_ids[0], skip_special_tokens=True)

print("Comparación y recomendación:\n", comparison)

Comparación y recomendación:
 # Seaborn: statistical data visualization # Seaborn is a Python data visualization library based on matplotlib . It provides a high-level interface for drawing attractive and informative statistical graphics in Python. Matplotlib: Visualization with Python Skip to main content Ctrl + K Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Matplotlib 3.9.0 Released We thank the 175 authors for the 450 pull requests that comprise the 3.9.0 release. Older Announcements Resources # Be sure to check the Users guide and the API docs . Join our community at discourse.matplotlib.org to get help, share your work, and discuss contributing & development. Check out the Matplotlib tag on StackOverflow to get help, share your work, and discuss contributing & development.


In [13]:
def chunk_text(text, max_tokens=400):
    words = text.split()
    return [' '.join(words[i:i+max_tokens]) for i in range(0, len(words), max_tokens)]

def summarize_long_text(text):
    chunks = chunk_text(text)
    summaries = [summarize(chunk) for chunk in chunks]
    return ' '.join(summaries)


In [ ]:
def agente_ia(urls):
    textos = [get_page_text(url) for url in urls]
    resumenes = [summarize_long_text(texto) for texto in textos]
    prompt_comparacion = construir_prompt(resumenes)
    return generar_respuesta(prompt_comparacion)


In [16]:
class VisualizationAgentFSM:
    def __init__(self, urls):
        self.urls = urls
        self.state = "buscar"
        self.texts = []
        self.summaries = []
        self.prompt = ""
        self.response = ""

    def run(self):
        while self.state != "final":
            if self.state == "buscar":
                self.texts = [get_page_text(url) for url in self.urls]
                self.state = "resumir"
            elif self.state == "resumir":
                self.summaries = [summarize_long_text(text) for text in self.texts]
                self.state = "generar_prompt"
            elif self.state == "generar_prompt":
                self.prompt = self.build_prompt()
                self.state = "responder"
            elif self.state == "responder":
                self.response = self.generate_response(self.prompt)
                self.state = "final"
        return self.response

    def build_prompt(self):
        return f"""
Compare objectively the following three data visualization libraries:

Seaborn: {self.texts[0]}
Matplotlib: {self.texts[1]}
Plotly: {self.texts[2]}

Create a markdown table with the following criteria:
- Ease of use
- Visual aesthetics
- Customization
- Interactivity
- Statistical support
- Integration

Give recommendations on when to use each, and a final general-purpose recommendation.
"""

    def generate_response(self, prompt):
        inputs = tokenizer.encode(prompt, return_tensors="pt", max_length=1024, truncation=True)
        response_ids = model.generate(inputs, max_length=300, num_beams=4, early_stopping=True)
        return tokenizer.decode(response_ids[0], skip_special_tokens=True)


In [17]:
urls = [
    "https://seaborn.pydata.org/",
    "https://matplotlib.org/",
    "https://plotly.com/python/"
]

agente = VisualizationAgentFSM(urls)
respuesta_final = agente.run()
print(respuesta_final)


Seaborn: statistical data visualization — seaborn 0.13.2 documentation Ctrl + K Installing Gallery Tutorial API Releases Citing FAQ GitHub StackOverflow Twitter Site Navigation Installing Gallery Tutorial API Releases Citing FAQ GitHub StackOverflow Twitter Matplotlib: Visualization with Python Skip to main content Ctrl + K Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Matplotlib 3.9.0 Released We thank the 175 authors for the 450 pull requests that comprise the 3.9.0 release. Older Announcements Resources # Join our community at discourse.matplotlib.org to get help, share your work, and discuss contributing & development. Check out the Matplotlib tag on StackOverflow to get help, share your work, and discuss contributing & development. Check out the Matplotlib tag on StackOverflow to get help, share your work, and discuss contr

In [15]:
respuesta_final

'Summary: Create a markdown table for Plotly products.'

In [18]:
from langchain.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def scrape_website(url: str) -> str:
    """Devuelve el texto principal de una página web dada la URL"""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    texts = soup.stripped_strings
    return " ".join(texts)


In [3]:
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

In [19]:
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

# URLs de ejemplo
urls = [
    "https://seaborn.pydata.org/",
    "https://matplotlib.org/",
    "https://plotly.com/python/"
]

# 🔍 Función para obtener el texto de cada web
def get_page_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    texts = soup.stripped_strings
    return ' '.join(texts)

# Obtenemos los textos
texts = [get_page_text(url) for url in urls]


In [20]:
# ✨ Cambia aquí el modelo según tu necesidad:
model_name = "google/flan-t5-base"  # o "gpt2", "distilgpt2", etc.

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Detectar tipo de modelo
is_seq2seq = "t5" in model_name or "flan" in model_name or "mt5" in model_name

# Cargar modelo según el tipo
if is_seq2seq:
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
else:
    model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [21]:
def generate_output(prompt, max_length=3000):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    if is_seq2seq:
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
    else:
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [22]:
def generate_output(prompt, max_length=300):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    if is_seq2seq:
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
    else:
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [24]:
# Summarize each website
summaries = [generate_output("Summarize: " + text[:1000]) for text in texts]

# Create the comparison message
comparison_prompt = (
"Here are three visualization libraries:\n\n"
f"1. Seaborn: {texts[0]}\n"
f"2. Matplotlib: {texts[1]}\n"
f"3. Plot: {texts[2]}\n\n"
"Compare them in a table considering:\n"
"- Ease of use\n- Aesthetics\n- Personalization\n"
"- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n"
"Make a final recommendation for general use."
)

# Get model response
comparison = generate_output(comparison_prompt)
print("\n🔍 Comparison and recommendation:\n", comparison)



🔍 Comparison and recommendation:
 Seaborn: statistical data visualization # Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics in Python. Matplotlib: Visualization with Python Skip to main content Ctrl + K Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Plot types User guide Tutorials Examples Reference Contribute Releases Gitter Discourse GitHub Twitter Matplotlib 3.9.0 Released We thank the 175 authors for the 450 pull requests that comprise the 3.9.0 release. Older Announcements Resources # Join our community at discourse.matplotlib.org to get help, share your work, and discuss contributing & development. Check out the Matplotlib tag on StackOverflow to get help, share your work, and discuss contributing & development. Check out the Matplotlib tag on StackOverflow to get help, share your work, and discuss contributing

In [2]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings

In [33]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings

# 1. Carga de modelos ligeros
generator = pipeline("text2text-generation", model="google/flan-t5-small")
llm = HuggingFacePipeline(pipeline=generator)

comparison_prompt = (
"Here are three visualization libraries:\n\n"
"Compare them in a table considering:\n"
"- Ease of use\n- Aesthetics\n- Personalization\n"
"- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n"
"Make a final recommendation for general use."
)

#embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# Embeddings
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# Crear base vectorial FAISS
vector_db = FAISS.from_texts(texts=comparison_prompt, embedding=embedding_model)

Device set to use cpu


In [51]:
class AutonomousAgentFSM:
    def __init__(self, goal, llm, retriever):
        self.state = "planear"
        self.goal = goal
        self.subtasks = []
        self.results = []
        self.evaluation = ""
        #self.qa_chain = RetrievalQA(llm=llm, retriever=retriever)
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=retriever,
            chain_type="stuff"  # o "map_reduce", "refine", etc., según tu necesidad
        )
        

    def run(self):
        while self.state != "final":
            if self.state == "planear":
                self.subtasks = self.plan()
                self.state = "ejecutar"
            elif self.state == "ejecutar":
                self.results = [self.execute(subtask) for subtask in self.subtasks]
                self.state = "evaluar"
            elif self.state == "evaluar":
                self.evaluation = self.evaluate(self.results)
                self.state = "final"
        return self.evaluation

    def plan(self):
        print(f"[PLAN] Objective: {self.goal}")
        return [
            #f"Searching information about {self.goal}",
            "Compare the three libraries",
            #"Create a markdown table with the comparison",
            "Recommend the best one for general use"
            
        ]

    def execute(self, subtask):
        print(f"[EXECUTE] Subtarea: {subtask}")
        return self.qa_chain.run(subtask)

    def evaluate(self, results):
        print(f"[EVALUATE] Resultados: {results}")
        return f"Evaluación completada. Resultados:\n" + "\n".join(results)


In [69]:
class AutonomousAgentFSM:
    def __init__(self, goal, llm, retriever):
        self.goal = goal
        self.llm = llm
        self.retriever = retriever
        self.qa_chain = RetrievalQA.from_chain_type(llm=self.llm, retriever=self.retriever, chain_type="stuff")
        self.state = "planificar"
        self.subtasks = []
        self.results = []
        self.evaluation = ""

    def plan(self):
        # Puedes hacer esto dinámico o usar prompts para generar subtareas
        return [
            f"Search information about {self.goal}",
            "Compare the three libraries",
            #"Create a markdown table with the comparison",
            "Recomend the best one for general use",
            "Explain your choice"
        ]
    #def plan(self):
    #    prompt = f"Divide the following objective into clearly and orderly subtasks: {self.goal}"
    #    plan_output = self.llm(prompt)
    #    subtasks = [line.strip("-• ") for line in plan_output.split("\n") if line.strip()]
    #    return subtasks


    def execute(self, task):
        print(f"🔧 Ejecutando subtarea: {task}")
        return self.qa_chain.run(task)

    def evaluate(self, results):
        return f"✅ Evaluación de resultados:\n- " + "\n- ".join(results)

    def run(self):
        while self.state != "final":
            if self.state == "planificar":
                self.subtasks = self.plan()
                self.state = "ejecutar"
            elif self.state == "ejecutar":
                for task in self.subtasks:
                    result = self.execute(task)
                    self.results.append(result)
                self.state = "evaluar"
            elif self.state == "evaluar":
                self.evaluation = self.evaluate(self.results)
                self.state = "final"
        return self.evaluation


In [4]:
# Preparación del entorno
from transformers import pipeline
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
# Preparación del entorno
from transformers import pipeline
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS

# Cargar modelos
generator = pipeline("text2text-generation", model="google/flan-t5-base") # o "google/flan-t5-small"
llm = HuggingFacePipeline(pipeline=generator)
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

urls = [
    "https://seaborn.pydata.org/",
    "https://matplotlib.org/",
    "https://plotly.com/python/getting-started/"
]

# 🔍 Función para obtener el texto de cada web
def get_page_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    texts = soup.stripped_strings
    return ' '.join(texts)

# Obtenemos los textos
texts = [get_page_text(url) for url in urls]

#print(texts[2])

vector_db = FAISS.from_texts(texts=texts, embedding=embedding_model)
retriever = vector_db.as_retriever()

prompt = (
"Compare the three visualization libraries considering:\n"
"- Ease of use\n- Aesthetics\n- Personalization\n"
"- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n"
"Make a final one recommendation for general use."
)

# Correr el agente FSM
agent_fsm = AutonomousAgentFSM(
    goal=prompt,
    llm=llm,
    retriever=retriever
)
output = agent_fsm.run()
print("\n=== Resultado final ===\n", output)


Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (3774 > 512). Running this sequence through the model will result in indexing errors


[PLAN] Objective: ['Compare the three visualization libraries considering:\n- Ease of use\n- Aesthetics\n- Personalization\n- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n', 'Make a final one recommendation for general use.']
[EXECUTE] Subtarea: Compare the three libraries
[EXECUTE] Subtarea: Recommend the best one for general use
[EVALUATE] Resultados: ['Matplotlib — Visualization with Python Matplotlib — Visualization with Python', 'Getting Started with Plotly']

=== Resultado final ===
 Evaluación completada. Resultados:
Matplotlib — Visualization with Python Matplotlib — Visualization with Python
Getting Started with Plotly


In [70]:
prompt = (
"Compare the three visualization libraries considering:\n"
"- Ease of use\n- Aesthetics\n- Personalization\n"
"- Interactivity\n- Statistical support\n- Jupyter/Dash integration\n\n"
"Make a final one recommendation for general use."
)

prompt = "Compare the three visualization libraries and choose the best one for general use."

prompt = "Compare the three visualization libraries and show if are easy of use, aesthetics, personalization, interactivity, statistical support, jupyter or dash integration and last choose the best one for general use."

prompt = (
    #"Given the following objective: 'Compare three data visualization libraries and "
    #"evaluate them in terms of ease of use, aesthetics, customization, interactivity, "
    #"evaluate them in terms of statistical support, integration with Jupyter or Dash, and select the best one for general use', "
    "choose one visualization library and explain why it is the best choice."
    #"Also, provide a markdown table comparing the three libraries based on the criteria mentioned."
)

# Correr el agente FSM
agent_fsm = AutonomousAgentFSM(
    goal=prompt,
    llm=llm,
    retriever=retriever
)
output = agent_fsm.run()
print("\n=== Resultado final ===\n", output)

🔧 Ejecutando subtarea: Search information about choose one visualization library and explain why it is the best choice.
🔧 Ejecutando subtarea: Compare the three libraries
🔧 Ejecutando subtarea: Recomend the best one for general use
🔧 Ejecutando subtarea: Explain your choice

=== Resultado final ===
 ✅ Evaluación de resultados:
- Matplotlib — Visualization with Python
- Matplotlib — Visualization with Python Matplotlib — Visualization with Python
- Getting Started with Plotly
- You can find the following information in the following sources:
